# 1. The Mental Model: Sync vs. Async

To understand asyncio, you first have to understand the difference between blocking (synchronous) and non-blocking (asynchronous) operations. Python is traditionally synchronous.

The Kitchen Analogy:
Imagine a single chef making breakfast.

* Synchronous (Blocking): The chef puts bread in the toaster. They stand there, staring at the toaster for 2 minutes doing absolutely nothing until it pops. Then, they start frying the eggs.

* Asynchronous (Non-Blocking): The chef puts bread in the toaster. While it's toasting, they immediately turn around and start frying the eggs. When the toast pops, they plate it.

In backend engineering, the "toaster" is an I/O operation: a database query, a network request to an external API, or reading a file from disk. While the network is doing its job, your CPU is sitting idle. Async programming allows your Python application to handle other incoming requests instead of waiting idly.

# 2. The Event Loop
The Event Loop is the engine that makes async programming possible. It is a continuous loop that runs in a single thread.  Think of it as a highly efficient manager:
* It keeps track of all running tasks.
* If a task hits an I/O operation (like waiting for a MongoDB response), the task tells the event loop, "I'm going to be waiting for a bit."
* The event loop pauses that task and instantly switches to another task that is ready to execute.  
* Once the I/O operation is finished, the event loop resumes the original task right where it left off.  

# 3. Coroutines (async def and await)
A coroutine is a specialized version of a Python function that can pause and resume its execution.
* async def: This defines a coroutine. Calling an async def function does not run it immediately. Instead, it returns a coroutine object.
* await: This is the keyword that tells the event loop, "Pause this coroutine here until the awaited operation finishes. Go do something else in the meantime." You can only use await inside an async def function.  

# 4. asyncio.run() and asyncio.sleep()
* asyncio.run(coro): This is the main entry point for an async program. It creates the event loop, runs the passed coroutine until it completes, and then closes the loop.  
* asyncio.sleep(seconds): This is a non-blocking delay. Unlike time.sleep() (which freezes the entire Python thread, completely defeating the purpose of async), asyncio.sleep() hands control back to the event loop so other code can run. It is heavily used in testing to mock slow database or API calls.  

In [2]:
import time
import asyncio

# ==========================================
# 1. THE SYNCHRONOUS WAY (Blocking)
# ==========================================
def sync_db_query(query_id):
    print(f"Sync: Starting DB Query {query_id}")
    time.sleep(2)  # Simulates a slow network/DB call. THE ENTIRE THREAD STOPS HERE.
    print(f"Sync: Finished DB Query {query_id}")
    return f"Data {query_id}"

def run_sync():
    start = time.time()
    sync_db_query(1)
    sync_db_query(2)
    print(f"Sync Total Time: {time.time() - start:.2f}s\n")

# ==========================================
# 2. THE ASYNCHRONOUS WAY (Non-Blocking)
# ==========================================
async def async_db_query(query_id):
    print(f"Async: Starting DB Query {query_id}")
    # await hands control back to the event loop while we wait for the "DB"
    await asyncio.sleep(2) 
    print(f"Async: Finished DB Query {query_id}")
    return f"Data {query_id}"

async def run_async():
    start = time.time()
    
    # We will cover 'gather' in Phase 2, but this tells the event loop 
    # to run these two coroutines concurrently.
    await asyncio.gather(
        async_db_query(1),
        async_db_query(2)
    )
    print(f"Async Total Time: {time.time() - start:.2f}s\n")

# ==========================================
# EXECUTION
# ==========================================
if __name__ == "__main__":
    run_sync()
    
    # asyncio.run() sets up the event loop and executes our main coroutine
    # asyncio.run(run_async()) -> not allowed in notebook
    await run_async()

Sync: Starting DB Query 1
Sync: Finished DB Query 1
Sync: Starting DB Query 2
Sync: Finished DB Query 2
Sync Total Time: 4.00s

Async: Starting DB Query 1
Async: Starting DB Query 2
Async: Finished DB Query 1
Async: Finished DB Query 2
Async Total Time: 2.02s



Notice the timeline. The async version started both queries before either finished, cutting the total execution time in half.